# Doctor-Patient Matching: Segmented Model with CatBoost
## Scenario B: Predicting Composite Success Score

In [ ]:
import pandas as pd, numpy as np, catboost as cb, pgeocode, subprocess, sys
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler

# --- File paths and data loading ---
file_paths = {"patient": "Master_df_sample.xlsx - Patient_df.csv", "encounter": "Master_df_sample.xlsx - Encounter_df.csv", "provider": "Master_df_sample.xlsx - Provider_df.csv", "hospital": "Master_df_sample.xlsx - Hospital_df.csv"}
patient_df, encounter_df, provider_df, hospital_df = (pd.read_csv(file_paths[k]) for k in file_paths)
print("Dataframes loaded successfully.")

In [ ]:
# --- Feature Engineering ---
master_df = pd.merge(pd.merge(pd.merge(encounter_df, patient_df, on='patient_id'), provider_df, on='provider_id'), hospital_df, left_on='hospital_affiliation', right_on='hospital_id', how='left')
master_df['race_match'] = (master_df['race'] == master_df['provider_race']).astype(int)
master_df['ethnicity_match'] = (master_df['ethnicity'] == master_df['provider_ethnicity']).astype(int)
master_df['language_match'] = (master_df['language_match'] == True).astype(int)
master_df['distance_km'] = pgeocode.GeoDistance('US').query_postal_code(master_df['zip_code'].astype(str).tolist(), master_df['zip_code_hosp'].astype(str).tolist())
mean_dist_by_specialty = master_df.groupby('specialty')['distance_km'].transform('mean')
master_df['distance_km'].fillna(mean_dist_by_specialty, inplace=True); master_df['distance_km'].fillna(master_df['distance_km'].mean(), inplace=True)
min_distance, max_distance = master_df['distance_km'].min(), master_df['distance_km'].max()
master_df['proximity_score'] = 1 - ((master_df['distance_km'] - min_distance) / (max_distance - min_distance))
print("Feature engineering complete.")

In [ ]:
# --- Target, Feature Definition, and Training (CatBoost) ---
scaler = MinMaxScaler()
master_df[['satisfaction_norm', 'adherence_norm']] = scaler.fit_transform(master_df[['patient_satisfaction', 'treatment_adherence']])
master_df['success_score'] = (master_df['adherence_norm'] * 0.5 + master_df['satisfaction_norm'] * 0.5)
target = 'success_score'
master_df.rename(columns={'cultural_competency_rating_y': 'cultural_competency_rating_prov'}, inplace=True)
features = ['years_experience', 'cultural_competency_rating_prov', 'communication_rating', 'race_match', 'ethnicity_match', 'language_match', 'proximity_score', 'interpreter_services_24_7']
unique_preferences = master_df['cultural_preferences'].unique()
trained_models, learned_weights_dict, best_hyperparameters, test_metrics_dict = {}, {}, {}, {}

for i, preference in enumerate(unique_preferences):
    print(f"\n--- Training model for preference: {preference} ({i+1}/{len(unique_preferences)}) ---")
    segment_df = master_df[master_df['cultural_preferences'] == preference].copy()
    if len(segment_df) < 100: continue
    X_segment, y_segment = segment_df[features], segment_df[target]
    X_train_val, X_test, y_train_val, y_test = train_test_split(X_segment, y_segment, test_size=0.2, random_state=42)
    param_grid = {'iterations': [100, 200, 300], 'learning_rate': [0.01, 0.05, 0.1], 'depth': [4, 6, 8], 'l2_leaf_reg': [1, 3, 5, 7]}
    cbr = cb.CatBoostRegressor(random_state=42, verbose=0)
    random_search = RandomizedSearchCV(estimator=cbr, param_distributions=param_grid, n_iter=50, cv=5, verbose=0, random_state=42, scoring='neg_mean_squared_error')
    print(f'Starting hyperparameter tuning on {len(X_train_val)} samples...')
    random_search.fit(X_train_val, y_train_val)
    print('Tuning complete. Best parameters found:', random_search.best_params_)
    best_model = random_search.best_estimator_
    final_predictions = best_model.predict(X_test)
    rmse, mae, r2 = np.sqrt(mean_squared_error(y_test, final_predictions)), mean_absolute_error(y_test, final_predictions), r2_score(y_test, final_predictions)
    print(f"\nFinal Evaluation -> RMSE: {rmse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}")
    test_metrics = {'rmse': rmse, 'mae': mae, 'r2': r2, 'test_set_size': len(X_test)}
    learned_weights = pd.Series(best_model.feature_importances_, index=features).sort_values(ascending=False)
    test_metrics_dict[preference], trained_models[preference], learned_weights_dict[preference], best_hyperparameters[preference] = test_metrics, best_model, learned_weights, random_search.best_params_
    print(f"--- Best model for '{preference}' trained and stored ---")